# Search Algorithm Evaluation
This notebook computes NDCG@5, Precision@5, and MRR for the BM25, Dense, and Hybrid search algorithms using the manually graded `qrels.csv`.

In [2]:
import pandas as pd
import numpy as np
import time
from search_engine import BM25Searcher, DenseSearcher, HybridSearcher

# --- CONFIGURATION ---
# Threshold for binary metrics (Precision/MRR). 
# Relevance >= THRESHOLD is treated as 1, else 0.
BINARY_THRESHOLD = 3
TOP_K = 5
# ---------------------

In [3]:
# Load models
print("Loading dataframe...")
movies = pd.read_pickle('movies_with_embeddings.pkl')
bm25_searcher = BM25Searcher(movies)
dense_searcher = DenseSearcher(movies)
hybrid_searcher = HybridSearcher(bm25_searcher, dense_searcher)

Loading dataframe...
Loading precalculated BM25 index from bm25_index.pkl...
BM25 Index loaded successfully.
Loading SentenceTransformer model (paraphrase-multilingual-MiniLM-L12-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading precalculated FAISS index from faiss_index.bin...
FAISS Index and ID map loaded successfully.


In [4]:
# Load qrels
try:
    qrels_df = pd.read_csv('qrels_to_grade.csv')
except FileNotFoundError:
    print("Error: qrels_to_grade.csv not found. Please run 02_pooling_generation.ipynb first.")
    raise

# Ensure relevance is numeric
qrels_df['relevance'] = pd.to_numeric(qrels_df['relevance'], errors='coerce').fillna(0)

# Build ground truth dictionary
ground_truth = {}
for _, row in qrels_df.iterrows():
    q = row['query']
    t = row['movie_title']
    r = float(row['relevance'])
    if q not in ground_truth:
        ground_truth[q] = {}
    ground_truth[q][t] = r

queries = list(ground_truth.keys())
print(f"Loaded ground truth for {len(queries)} queries.")

Loaded ground truth for 50 queries.


In [5]:
def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances, dtype=float)[:k]
    if relevances.size:
        return np.sum(relevances / np.log2(np.arange(2, relevances.size + 2)))
    return 0.0

def evaluate_method(searcher_func, queries, ground_truth, k=5, threshold=3):
    ndcg_scores = []
    mrr_scores = []
    p_at_k_scores = []
    latencies = []
    
    for q in queries:
        start_time = time.time()
        results = searcher_func(q, top_k=k)
        end_time = time.time()
        latencies.append((end_time - start_time) * 1000) # ms
        
        # Calculate NDCG
        true_relevance = []
        for i, res in enumerate(results):
            title = res['title']
            rel = ground_truth[q].get(title, 0.0)
            true_relevance.append(rel)
            
        # IDCG Global usando todas las calificaciones en ground_truth para esta query
        all_query_rels = list(ground_truth[q].values())
        ideal_relevances = sorted(all_query_rels, reverse=True)[:k]
        idcg = dcg_at_k(ideal_relevances, k)
        
        if idcg > 0:
            dcg = dcg_at_k(true_relevance, k)
            ndcg = dcg / idcg
        else:
            ndcg = 0.0
        ndcg_scores.append(ndcg)
        
        # Calculate MRR and Precision@K
        binary_relevance = [1 if r >= threshold else 0 for r in true_relevance]
        # Pad with 0 if results < k
        while len(binary_relevance) < k:
            binary_relevance.append(0)
            
        p_at_k = np.sum(binary_relevance) / k
        p_at_k_scores.append(p_at_k)
        
        rr = 0.0
        for idx, val in enumerate(binary_relevance):
            if val == 1:
                rr = 1.0 / (idx + 1)
                break
        mrr_scores.append(rr)
        
    return {
        'NDCG@5': np.mean(ndcg_scores),
        'Precision@5': np.mean(p_at_k_scores),
        'MRR': np.mean(mrr_scores),
        'Latencia (ms)': np.mean(latencies)
    }

# Evaluate all three
print("Evaluating BM25...")
bm25_metrics = evaluate_method(bm25_searcher.search, queries, ground_truth, TOP_K, BINARY_THRESHOLD)
print("Evaluating Dense...")
dense_metrics = evaluate_method(dense_searcher.search, queries, ground_truth, TOP_K, BINARY_THRESHOLD)
print("Evaluating Hybrid...")
hybrid_metrics = evaluate_method(hybrid_searcher.search, queries, ground_truth, TOP_K, BINARY_THRESHOLD)


Evaluating BM25...
Evaluating Dense...
Evaluating Hybrid...


In [7]:
# Create results dataframe
results_df = pd.DataFrame([
    {'Method': 'BM25', **bm25_metrics},
    {'Method': 'Dense', **dense_metrics},
    {'Method': 'Hybrid', **hybrid_metrics}
])

print("\n=== FINAL EVALUATION RESULTS ===")
display(results_df.round(4))



=== FINAL EVALUATION RESULTS ===


,Method,NDCG@5,Precision@5,MRR,Latencia (ms)
0,BM25,0.6037,0.456,0.6733,43.5325
1,Dense,0.4801,0.348,0.6070,106.0315
2,Hybrid,0.5608,0.424,0.6740,101.0732
